# Training methodology decisions: HPO, QLoRA/QAT, and quantization dtypes

This notebook documents three methodology decisions made for text2bash's
Phase 3 (fine-tuning) and Phase 4.1 (post-training quantization), each
backed by a quick literature check rather than assumption. See `PLAN.md`
for how these fit into the overall project.

## 1. Why there's no LoRA rank/alpha hyperparameter search

The honest starting position: rank=8, alpha=16 in `train/finetune_lora.py`
were picked because they're the standard starting pair everyone uses, not
because they were searched. The question is whether that's good enough or
whether a sweep would have materially changed the result.

**The most load-bearing source here is Sebastian Raschka's LoRA ablation
work** — not a single arXiv paper, but hundreds of controlled fine-tuning
runs (built on the Lit-GPT codebase he co-maintains at Lightning AI),
written up in [*Practical Tips for Finetuning LLMs Using LoRA (Low-Rank
Adaptation)*](https://magazine.sebastianraschka.com/p/practical-tips-for-finetuning-llms)
and the companion FAQ [*How do rank and alpha affect LoRA behavior in
practice?*](https://sebastianraschka.com/faq/docs/lora-rank-alpha.html).
This is the piece most of the "rank/alpha barely matters" folk wisdom
traces back to, so it's worth citing directly rather than the blog posts
that paraphrase it. Two findings from that work are the actual basis for
skipping a rank/alpha grid here:

- Across his hundreds of runs, **benchmark results were consistent across
  different random seeds/runs** — the variance from randomness alone is in
  the same ballpark as the variance from changing rank. A rank sweep on a
  single run risks chasing noise.
- His own heuristic, which this project already follows, is **alpha = 2 ×
  rank** — not something to search per-run, just a fixed ratio.

This is corroborated, less directly, by two arXiv papers surfaced in the
same search: [*LoRA Learns Less and Forgets Less*](https://arxiv.org/pdf/2405.09673)
(rank mostly affects how much of the base model's prior behavior survives
fine-tuning, not task accuracy at the scale used here) and [*Post-Training
Science for Supervised Fine-Tuning*](https://arxiv.org/pdf/2609.01244)
(learning rate dominates over architectural choices like rank once a
reasonable rank is picked).

**What this project did instead of a rank/alpha sweep:** nothing — rank=8,
alpha=16 were kept as-is. What it did do instead, because the evidence
specifically flags it as the lever that actually matters, is a small
learning-rate sweep (1e-4 / 2e-4 / 4e-4) over a fixed rank/alpha. Results
are in `PLAN.md`'s Phase 3.1 section once that run completes.

## 2. QLoRA and QAT: why neither was used for training

*(Written the way I'd actually explain it to a teammate, not the way a
vendor blog post would.)*

Let's not kid ourselves about what QLoRA is for. It exists to solve one
problem: your model doesn't fit in your GPU's memory in full precision.
That's it. That's the whole pitch. Dettmers didn't build QLoRA because
4-bit weights are magically better than 16-bit ones — he built it because
people were trying to fine-tune 65B-parameter models on a single 48GB
card and the numbers didn't work out otherwise.

Now look at what we actually have: a 596M-parameter model, loaded in
bf16, sitting at a bit over 1GB of VRAM, on an 8GB card with 6.7GB free.
We are not memory-constrained. Not even close. Running QLoRA here would
mean paying the dequantization tax on every forward and backward pass —
real, measured overhead — to solve a memory problem we don't have. That's
not efficient, that's cargo-culting a Reddit thread about fine-tuning
Llama-70B onto a problem two orders of magnitude smaller. If this were a
7B+ model on this same 8GB card, this section would read completely
differently and QLoRA would be the obvious call. It isn't, so it's not.

QAT is a different kind of "no." QLoRA was "solves a problem we don't
have." QAT is "solves a problem we haven't measured yet." The entire
point of quantization-aware training is to claw back accuracy that
post-training quantization loses, by making the model's weights learn to
be quantization-friendly during training itself. That's expensive —
you're threading gradients through quantized weights, which is fiddly and
slow compared to a normal fine-tune. Paying that cost before you know
whether you need it is backwards. Phase 4.1 runs the actual post-training
quantization and measures the actual accuracy drop at fp8, int8, int4,
and fp4. If one of those drops is bad enough to matter, *then* QAT is a
reasonable next move, armed with a number instead of a guess. Doing it
preemptively would be optimizing against a problem that might not exist.

So: no QLoRA, because there's no memory problem to solve. No QAT, because
there's no measured accuracy problem yet — and Phase 4.1 is exactly the
step that finds out.

## 3. Quantization dtypes for Phase 4.1: technique and hardware support

Phase 4.1 asked for a post-training quantization sweep across fp8, fp4,
int8, int4, and nvfp4. These five names get thrown around as if they're
interchangeable "pick a bit width" options, but they're actually three
different *techniques* with different hardware requirements — worth being
precise about before running anything, since "unsupported" means
different things for different ones (one is a software fallback with no
speed benefit; another has no fallback path at all on this GPU).

This machine's GPU is an **RTX 3070 — Ampere architecture, compute
capability 8.6** (confirmed via `nvidia-smi` / CUDA docs during this
session).

| dtype | technique / library | what it actually requires | this GPU |
|---|---|---|---|
| **fp8** (e4m3) | `torchao` weight-only quantization — weights stored in fp8, upcast to bf16 at matmul time | Nothing, strictly — this is a software upcast path. Native fp8 **tensor cores** (for a real throughput win) arrived with Ada Lovelace / Hopper | ✅ runs, but **emulated**: measures accuracy impact only, no speed benefit (Ampere has no fp8 tensor cores) |
| **fp4** | `bitsandbytes` custom 4-bit kernels (`bnb_4bit_quant_type="fp4"`) | bnb's own hand-written CUDA kernels — this was never a hardware tensor-core format on *any* GPU generation, including current ones. It's a library-level scheme, not silicon | ✅ fully supported, runs natively |
| **int8** | `bitsandbytes` LLM.int8() (mixed-precision decomposition with outlier handling) | bnb's own CUDA kernels | ✅ fully supported, runs natively |
| **int4** | `torchao` weight-only int4 (true affine integer quantization, groupwise, tinygemm kernels) | SM80+ (Ampere or newer) for the fused kernel | ✅ fully supported — Ampere (SM86) clears the bar |
| **nvfp4** | NVIDIA's native FP4 format (E2M1, two-level microscaling), a genuine Tensor Core instruction | **Blackwell silicon only** (B100/B200, RTX 50-series) — confirmed no pre-Blackwell GPU, including the A100/H100, has FP4 tensor cores | ❌ **excluded from the sweep.** No hardware path, and no practical software-emulation path exists in current tooling either (unlike fp8, which has a usable weight-only fallback) |

**Bottom line for the Phase 4.1 sweep:** bf16 (reference), fp8, fp4, int8,
and int4 all run on this hardware and are included. nvfp4 is a Blackwell-only
format with no fallback worth building for a single project — it's
documented here rather than silently dropped, but it's out of scope.